# TikzTable: styling and drawing

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

This page continues from [TikzTable basics](tikztable.ipynb). `TikzTable` supports every
formatter and table-level option shown in the [TexTable guide](textable.ipynb), then adds a
TikZ lattice for formatting that depends on cell geometry: framing selected cells, drawing
across several cells, and controlling the space around their contents.

In [ ]:
import numpy as np
import pandas as pd

from gerrytools.latex import TikzTable

districts = pd.DataFrame(
    {
        "District": [f"CD {i}" for i in range(1, 9)],
        "BVAP share": [0.12, 0.18, 0.22, 0.31, 0.38, 0.44, 0.52, 0.58],
        "Dem share": [0.35, 0.41, 0.44, 0.47, 0.50, 0.55, 0.61, 0.66],
        "Polsby-Popper": [0.18, 0.22, 0.27, 0.31, 0.33, 0.35, 0.41, 0.44],
        "Pop. deviation": [0.004, 0.002, np.nan, 0.006, 0.001, 0.008, 0.003, 0.005],
    }
)
districts

## Cell borders

`set_cell_border(row, col, sides)` uses 1-based coordinates in the rendered table. In this
example, row 1 is the column header, row 2 is DataFrame row 0, and column 1 is `District`.
A group header adds another rendered row, while `include_index()` adds a leading column.
The accepted sides are `"top"`, `"bottom"`, `"left"`, `"right"`, and `"all"`.

> <span class="notebook-admonition-title">Coordinate systems</span>
>
> `highlight_rows()` uses zero-based DataFrame row positions. `set_cell_border()` and
> `add_draw()` use one-based rendered coordinates because they operate after headers and
> index columns have been laid out.

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.set_cell_space_limits("2pt")
table.highlight_rows([3, 4], color="lightblue!20!white")
table.set_cell_border(5, [2, 3, 4, 5], "top")
table.set_cell_border(6, [2, 3, 4, 5], "bottom")
table.set_cell_border([5, 6], 2, "left")
table.set_cell_border([5, 6], 5, "right")
print(table)

![A bordered range over two highlighted rows][tikztable-borders]

[tikztable-borders]: ../../_static/images/latex/tikztable-borders.png

Passing row or column lists applies the requested sides to every selected cell. The four
calls above draw only the outside of the range. Use `"all"` when each cell needs its own
box. `set_cell_space_limits()` increases the minimum vertical breathing room without
changing the DataFrame or wrapping its values.

## Combine formatters with cell geometry

The shared formatter pipeline still controls *which* cells receive formatting. TikZ then
paints those fills across the complete cell. For `diverging_gradient_formatter()`, pass
`command_name=None` to select that full-cell path. Fixing `lo`, `mid`, and `hi` keeps the
colors comparable across several tables rather than rescaling them to each DataFrame.

`compose_formatters()` runs from right to left: this example first renders two decimal
places, then chooses a background from the original numeric value.

In [ ]:
from gerrytools.latex.commands import tex_twocolor_gradient_command
from gerrytools.latex.formatters import (
    compose_formatters,
    diverging_gradient_formatter,
    highlight_ge,
    round_decimals,
    wrap_with_tex_command,
)

table = TikzTable(districts)
table.set_decimal_count(2)
table.set_column_formatter(
    "Dem share",
    compose_formatters(
        diverging_gradient_formatter(
            lo=0.35,
            mid=0.50,
            hi=0.65,
            color_lo="alizarin",
            color_mid="white",
            color_hi="denim",
            command_name=None,
        ),
        round_decimals(2),
    ),
)
print(table)

![Diverging gradient in a TikzTable][tikztable-diverging]

[tikztable-diverging]: ../../_static/images/latex/tikztable-diverging.png

### Format every numeric cell

A column formatter is appropriate when one measure has a meaningful scale. A number
formatter applies to every numeric cell, which is useful for a compact diagnostic matrix.
Missing values bypass the formatter and use the string configured by `set_nan_string()`.

In [ ]:
table = TikzTable(districts)
table.set_nan_string("---")
table.set_number_formatter(compose_formatters(wrap_with_tex_command("heatmap"), round_decimals(2)))
table.document.add_command(tex_twocolor_gradient_command("heatmap"))
print(table)

![Two-color heatmap in a TikzTable][tikztable-heatmap]

[tikztable-heatmap]: ../../_static/images/latex/tikztable-heatmap.png

## Draw across cells

Use `add_draw()` when the annotation spans a range or expresses a relationship that cannot
be described as individual cell sides. It appends raw TikZ to the table's `\CodeAfter`
block. `set_table_name()` supplies the node prefix, and each cell is addressable as
`(<name>-<row>-<col>)` through the usual TikZ anchors. Give tables distinct names when
several are pasted into the same document so their nodes cannot collide.

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.set_table_name("diagnostics")
table.document.add_color("framecolor", "denim")
table.add_draw(
    r"\draw[framecolor, rounded corners=2pt, line width=0.8pt] "
    r"(diagnostics-5-2.north west) rectangle "
    r"(diagnostics-6-5.south east);"
)
print(table)

![A rounded frame drawn around a block of cells][tikztable-draws]

[tikztable-draws]: ../../_static/images/latex/tikztable-draws.png

The command uses the north-west anchor of one cell and the south-east anchor of another,
so the frame follows the table layout automatically. The boundary lattice is also available
as `(row-<i>)` and `(col-<j>)`. `clear_extra_draws()` removes the raw drawing commands, just
as `clear_cell_borders()` removes the higher-level cell borders.

## Put the pieces together

A finished table can combine the shared report formatting with TikZ-specific geometry.
Here, grouped headers organize the measures, a fixed diverging scale shows vote share, and
a threshold formatter identifies larger population deviations. The borders pick out those
same cells after the extra group-header row has shifted the rendered coordinates.

In [ ]:
table = TikzTable(districts)
table.set_header_groups(
    {
        "": ["District"],
        "Representation": ["BVAP share", "Dem share"],
        "Diagnostics": ["Polsby-Popper", "Pop. deviation"],
    }
)
table.set_cell_space_limits("1.5pt")
table.set_column_formatter(
    "Dem share",
    compose_formatters(
        diverging_gradient_formatter(
            lo=0.35,
            mid=0.50,
            hi=0.65,
            color_lo="alizarin",
            color_mid="white",
            color_hi="denim",
            command_name=None,
        ),
        round_decimals(2),
    ),
)
table.set_column_formatter(
    "Pop. deviation",
    compose_formatters(
        highlight_ge(0.006, color="amber!35!white"),
        round_decimals(3),
    ),
)
table.set_cell_border([6, 8], 5, "all")
table.add_toprule()
table.add_bottomrule()
print(table)

![A complete TikzTable with grouped headers, gradients, highlights, and borders][tikztable-showcase]

[tikztable-showcase]: ../../_static/images/latex/tikztable-showcase.png

## Choose the narrowest formatting tool

- Use `highlight_rows()` for a complete DataFrame row.
- Use a formatter when the displayed value determines the treatment.
- Use `set_cell_border()` for exact cells or rectangular boundaries.
- Use `add_draw()` only for geometry that crosses cells or needs other TikZ primitives.

These layers compose: formatters and row highlights are resolved before borders and raw
draw commands are added over the finished table.

## Related

- [TikzTable basics](tikztable.ipynb)
- [TexTable](textable.ipynb)
- [LaTeX API](../../api/latex.rst)